In [37]:
import pandas as pd

edges = pd.read_csv("datasets/edges.csv", dtype={"section":"string"})
nodes = pd.read_csv('datasets/nodes.csv')
characters = pd.read_csv('datasets/characters.csv')
persons = pd.read_csv('datasets/persons.csv')
families = pd.read_csv('datasets/family.csv')
marriages = pd.read_csv('datasets/marriages.csv')
unconnected_characters = characters[~characters['name'].isin(nodes['id'])]
unconnected_characters


,id,name,description,type,domain,gender,aliases,residence,theoi_url,level1,level2,level3,level4,level5
0,Q304227,Aba,Najad in Greek mythology,naiad,[],female,[],NaN,https://www.theoi.com/Nymphe/NympheAba.html,Mythological Greek Character,Greek deity,Greek nymph,naiad,NaN
9,Q2822505,Acanthis,daughter of Hippodamia and Autonous and sister...,Mythological Greek Character,[],female,['Acanthyllis'],NaN,NaN,Mythological Greek Character,NaN,NaN,NaN,NaN
13,Q605797,Aceso,Greek goddess of healing,Greek deity,[],female,[],NaN,https://www.theoi.com/Ouranios/AsklepiasAkeso....,Mythological Greek Character,Greek deity,NaN,NaN,NaN
21,Q340545,Achlys,primordial deity in Greek mythology,Greek primordial deity,[],female,['Akhlys'],NaN,https://www.theoi.com/Daimon/Akhlys.html,Mythological Greek Character,Greek deity,Greek primordial deity,NaN,NaN
29,Q356321,Adephagia,Greek goddess and personification of gluttony,allegorical Greek deity,[],female,[],NaN,https://www.theoi.com/Daimon/Adephagia.html,Mythological Greek Character,Greek deity,allegorical Greek deity,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1013,Q659437,Thalassa,"primordial sea goddess in Greek mythology, dau...",Greek primordial deity,[],female,[],NaN,https://www.theoi.com/Protogenos/Thalassa.html,Mythological Greek Character,Greek deity,Greek primordial deity,NaN,NaN
1042,Q2062262,Tiburtine Sibyl,Roman Sibyl,sibyl,[],female,['Albvnea'],NaN,NaN,Mythological Greek Character,sibyl,NaN,NaN,NaN
1044,Q2525222,Tinjis,Berber god,Mythological Greek Character,[],female,"['Tingis', 'Tinga']",NaN,NaN,Mythological Greek Character,NaN,NaN,NaN,NaN
1067,Q483760,Venus Anadyomene,one of the iconic representations of Aphrodite,Greek deity,[],NaN,"['The Burlesque of Venus', 'Anadyomene', 'birt...",NaN,NaN,Mythological Greek Character,Greek deity,NaN,NaN,NaN


In [38]:
characters['level1'].unique()

array(['Mythological Greek Character'], dtype=object)

In [39]:
persons = persons.drop(['Birth date', 'Birth place', 'Death place', 'Burial place'], axis=1)

In [40]:
# Step 1: Families + Marriages
family_parents = families.merge(marriages, left_on="Family", right_on="Marriage", how="left")

# Step 2: Build mapping of child -> parents
parent_map = family_parents[["Child", "Husband", "Wife"]].rename(
    columns={"Husband": "Father", "Wife": "Mother"}
)

# Step 3: Merge into Persons
persons_with_parents = persons.merge(parent_map, left_on="Person", right_on="Child", how="left")

# Drop helper column
persons_with_parents = persons_with_parents.drop(columns=["Child"])

# Create a lookup dictionary: ID -> Given
id_to_given = pd.Series(persons_with_parents.Given.values, index=persons_with_parents.Person).to_dict()

# Replace Father and Mother IDs with names
persons_with_parents["Father"] = persons_with_parents["Father"].map(id_to_given)
persons_with_parents["Mother"] = persons_with_parents["Mother"].map(id_to_given)

persons_with_parents

,Person,Given,Title,Gender,Father,Mother
0,[I0036],(11 other sons of Neleus),NaN,male,Neleus,Chloris (2)
1,[I0250],(15 other Nereids),NaN,female,Nereus,Doris (1)
2,[I0221],(2 children of Heracles and Megara),NaN,unknown,Heracles,Megara
3,[I0107],(8 children of Neoptolemus and Leonassa),NaN,unknown,Neoptolemus,Leonassa
4,[I0097],(8 other children of Priam and Hecuba),NaN,unknown,Priam,Hecuba
...,...,...,...,...,...,...
895,[I0372],Zephyrus,NaN,male,Aeolus,Eos
896,[I0372],Zephyrus,NaN,male,Astraeus,Eos
897,[I0616],Zethus,NaN,male,Zeus,Antiope (1)
898,[I0027],Zeus,NaN,male,Cronos,Rhea


In [41]:
persons_with_parents[persons_with_parents["Given"].str.contains(r"\(\d+\)$", regex=True, na=False)]


,Person,Given,Title,Gender,Father,Mother
40,[I0505],Abas (1),King (of Argos),male,Lynceus,Hypermnestra (1)
41,[I0709],Abas (2),NaN,male,Celeus,Metanira
42,[I0713],Abas (3),NaN,male,NaN,NaN
88,[I0522],Aethra (2),NaN,female,Pittheus,NaN
92,[I0411],Agenor (1),King (of Phoenicia),male,Poseidon,NaN
...,...,...,...,...,...,...
856,[I0678],Theoclymenus (1),NaN,male,Polypheides,NaN
857,[I0237],Theoclymenus (3),NaN,unknown,Thestor,NaN
865,[I0790],Thoas (1),King (of Lemnos),male,Dionysus,Ariadne
866,[I0790],Thoas (1),King (of Lemnos),male,Theseus,Ariadne


In [42]:
persons_with_parents['Given'] = persons_with_parents['Given'].str.replace(r'\s*\(\d+\)$', '', regex=True)
persons_with_parents['Father'] = persons_with_parents['Father'].str.replace(r'\s*\(\d+\)$', '', regex=True)
persons_with_parents['Mother'] = persons_with_parents['Mother'].str.replace(r'\s*\(\d+\)$', '', regex=True)
persons_with_parents.head(44)

,Person,Given,Title,Gender,Father,Mother
0,[I0036],(11 other sons of Neleus),NaN,male,Neleus,Chloris
1,[I0250],(15 other Nereids),NaN,female,Nereus,Doris
2,[I0221],(2 children of Heracles and Megara),NaN,unknown,Heracles,Megara
3,[I0107],(8 children of Neoptolemus and Leonassa),NaN,unknown,Neoptolemus,Leonassa
4,[I0097],(8 other children of Priam and Hecuba),NaN,unknown,Priam,Hecuba
5,[I0329],(997 other sons of Hypnos),NaN,male,Hypnos,NaN
6,[I0654],(A cloud-image of Hera),NaN,female,NaN,NaN
7,[I0061],(British House of Cornwall),NaN,male,"(Mythical kings of Britain; includes Brutus, L...",NaN
8,[I0290],(Dreams),NaN,unknown,Erebus,Nyx
9,[I0119],(Household servants of Priam),NaN,female,NaN,NaN


In [43]:
edges = edges.drop(['group_key', 'language', 'chapter', 'genre', 'years', 'section', 'myth', 'source_id', 'source_character', 'source_description', 'source_type', 'source_domain', 'source_gender', 'source_aliases', 'source_residence', 'source_theoi_url', 'source_level1', 'source_level2', 'source_level3', 'source_level4', 'source_level5', 'target_id', 'target_character', 'target_description', 'target_type', 'target_domain', 'target_gender', 'target_aliases', 'target_residence', 'target_theoi_url', 'target_level1', 'target_level2', 'target_level3', 'target_level4', 'target_level5'], axis=1)

In [44]:
edges

,source,target,normalized_weight,author,title,period
0,Aphrodite,Cronus,0.005291,Nonnus,Dionysiaca Bks 1 - 14,Late Roman
1,Aphrodite,Nereus,0.005291,Nonnus,Dionysiaca Bks 1 - 14,Late Roman
2,Aphrodite,Thetis,0.010582,Nonnus,Dionysiaca Bks 1 - 14,Late Roman
3,Aphrodite,Triton,0.010582,Nonnus,Dionysiaca Bks 1 - 14,Late Roman
4,Aphrodite,Zeus,0.015873,Nonnus,Dionysiaca Bks 1 - 14,Late Roman
...,...,...,...,...,...,...
48409,Orpheus,Rhodope,0.012121,Virgil,Eclogues,Imperial Roman
48410,Pallas,Paris,0.006061,Virgil,Eclogues,Imperial Roman
48411,Philomela,Tereus,0.012121,Virgil,Eclogues,Imperial Roman
48412,Phyllis,Zeus,0.012121,Virgil,Eclogues,Imperial Roman


In [45]:
characters = characters.drop(['id', 'type', 'residence', 'theoi_url', 'level1'], axis=1)

In [46]:
characters

,name,description,domain,gender,aliases,level2,level3,level4,level5
0,Aba,Najad in Greek mythology,[],female,[],Greek deity,Greek nymph,naiad,NaN
1,Abarbarea,water deity,[],female,[],Greek deity,Greek nymph,naiad,NaN
2,Abaris the Hyperborean,"semi-legendary Greek physician, magician, and ...",[],male,"['Abaris', 'the Hyperborean', 'Scythicus', 'Hy...",NaN,NaN,NaN,NaN
3,Abas,"mythical character, son of Lynceus",[],male,"['Abas', 'son of Lynceus', 'son of lynceus', '...",NaN,NaN,NaN,NaN
4,Abderus,son of Hermes in Greek mythology,[],male,['Abderos'],NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
1071,Zephyrus,"Greek god of the wind from the west, spring, f...",[],male,['Zephyr'],Greek deity,Anemoi,NaN,NaN
1072,Zethos,king in Greek mythology,[],male,['Zethus'],NaN,NaN,NaN,NaN
1073,Zeus,Greek god of the sky and king of the gods,"['lightning', 'thunder']",male,"['Dias', 'Jupiter', 'Dju Pater', 'Juppiter', '...",Greek deity,Olympian god,NaN,NaN
1074,Zeuxo,Oceanid of Greek mythology,[],female,[],Greek deity,Greek water deities,Oceanids,Okeanid


In [47]:
import ast
def ensure_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x) or x in ("[]", ""):
        return []
    try:
        return ast.literal_eval(x)  # safely parse string like "['alias1']"
    except (ValueError, SyntaxError):
        return [x]  # fallback: treat as single string

# ensure aliases column are lists of strings
characters["aliases"] = characters["aliases"].apply(ensure_list)

characters["all_names"] = characters.apply(lambda row: [row["name"]] + row["aliases"], axis=1)

# delete duplicate entries of lists in all_names column
characters['all_names'] = characters['all_names'].apply(lambda x: list(dict.fromkeys(x)))

characters_exploded = characters.explode("all_names")

merged_characters = characters_exploded.merge(persons_with_parents, left_on="all_names", right_on="Given", how="inner")

In [48]:
characters_exploded.head(25)

,name,description,domain,gender,aliases,level2,level3,level4,level5,all_names
0,Aba,Najad in Greek mythology,[],female,[],Greek deity,Greek nymph,naiad,NaN,Aba
1,Abarbarea,water deity,[],female,[],Greek deity,Greek nymph,naiad,NaN,Abarbarea
2,Abaris the Hyperborean,"semi-legendary Greek physician, magician, and ...",[],male,"[Abaris, the Hyperborean, Scythicus, Hyperbore...",NaN,NaN,NaN,NaN,Abaris the Hyperborean
2,Abaris the Hyperborean,"semi-legendary Greek physician, magician, and ...",[],male,"[Abaris, the Hyperborean, Scythicus, Hyperbore...",NaN,NaN,NaN,NaN,Abaris
2,Abaris the Hyperborean,"semi-legendary Greek physician, magician, and ...",[],male,"[Abaris, the Hyperborean, Scythicus, Hyperbore...",NaN,NaN,NaN,NaN,the Hyperborean
2,Abaris the Hyperborean,"semi-legendary Greek physician, magician, and ...",[],male,"[Abaris, the Hyperborean, Scythicus, Hyperbore...",NaN,NaN,NaN,NaN,Scythicus
2,Abaris the Hyperborean,"semi-legendary Greek physician, magician, and ...",[],male,"[Abaris, the Hyperborean, Scythicus, Hyperbore...",NaN,NaN,NaN,NaN,Hyperboreus
2,Abaris the Hyperborean,"semi-legendary Greek physician, magician, and ...",[],male,"[Abaris, the Hyperborean, Scythicus, Hyperbore...",NaN,NaN,NaN,NaN,Historicus
2,Abaris the Hyperborean,"semi-legendary Greek physician, magician, and ...",[],male,"[Abaris, the Hyperborean, Scythicus, Hyperbore...",NaN,NaN,NaN,NaN,Abaris Scythicus
2,Abaris the Hyperborean,"semi-legendary Greek physician, magician, and ...",[],male,"[Abaris, the Hyperborean, Scythicus, Hyperbore...",NaN,NaN,NaN,NaN,Abairis


In [49]:
merged_characters.head(30)

,name,description,domain,gender,aliases,level2,level3,level4,level5,all_names,Person,Given,Title,Gender,Father,Mother
0,Abas,"mythical character, son of Lynceus",[],male,"[Abas, son of Lynceus, son of lynceus, son of ...",NaN,NaN,NaN,NaN,Abas,[I0505],Abas,King (of Argos),male,Lynceus,Hypermnestra
1,Abas,"mythical character, son of Lynceus",[],male,"[Abas, son of Lynceus, son of lynceus, son of ...",NaN,NaN,NaN,NaN,Abas,[I0709],Abas,NaN,male,Celeus,Metanira
2,Abas,"mythical character, son of Lynceus",[],male,"[Abas, son of Lynceus, son of lynceus, son of ...",NaN,NaN,NaN,NaN,Abas,[I0713],Abas,NaN,male,NaN,NaN
3,Abderus,son of Hermes in Greek mythology,[],male,[Abderos],NaN,NaN,NaN,NaN,Abderus,[I0714],Abderus,NaN,male,Hermes,NaN
4,Absyrtus,mythical brother of Medea,[],male,"[Apsyrtus, Aegialeus]",NaN,NaN,NaN,NaN,Absyrtus,[I0514],Absyrtus,NaN,male,Aeetes,NaN
5,Absyrtus,mythical brother of Medea,[],male,"[Apsyrtus, Aegialeus]",NaN,NaN,NaN,NaN,Aegialeus,[I0741],Aegialeus,NaN,unknown,Inachus,Melia (individual representation of the Meliae)
6,Acacallis,daughter of Minos in Greek mythology,[],female,[Acacallis],NaN,NaN,NaN,NaN,Acacallis,[I0715],Acacallis,NaN,female,Minos,Pasiphae
7,Acamas,son of Theseus,[],male,[],NaN,NaN,NaN,NaN,Acamas,[I0671],Acamas,NaN,male,Theseus,Phaedra
8,Acarnan,son of Alcmaeon in Greek mythology,[],male,[],NaN,NaN,NaN,NaN,Acarnan,[I0721],Acarnan,NaN,male,Alcmaeon,Callirrhoe
9,Acastus,"mythical character, son of Pelias",[],male,[],NaN,NaN,NaN,NaN,Acastus,[I0253],Acastus,NaN,male,Pelias,Anaxibia


In [50]:
merged_characters = merged_characters.drop(['Person', 'Gender', 'all_names', 'Given'], axis=1)
merged_characters.to_csv('datasets/merged_characters.csv', index=False)

In [51]:
unique_characters = pd.read_csv('datasets/unique_characters.csv')

unique_characters["aliases"] = unique_characters["aliases"].apply(ensure_list)

# Step 1: Build a mapping from each name and alias → canonical name
name_to_canonical = {}

for _, row in unique_characters.iterrows():
    canonical = row["name"]
    name_to_canonical[canonical] = canonical
    for alias in row["aliases"]:
        name_to_canonical[alias] = canonical
print(name_to_canonical)

# Step 2: Replace Father/Mother with canonical names if alias exists
for col in ["Father", "Mother"]:
    unique_characters[col] = unique_characters[col].map(lambda x: name_to_canonical.get(x, x))

{'Abas': 'Abas', 'son of Lynceus': 'Abas', 'son of lynceus': 'Abas', 'son of Lynceos': 'Abas', 'Abantias': 'Abas', 'Abderus': 'Abderus', 'Abderos': 'Abderus', 'Absyrtus': 'Absyrtus', 'Apsyrtus': 'Absyrtus', 'Aegialeus': 'Absyrtus', 'Acacallis': 'Acacallis', 'Acamas': 'Acamas', 'Acarnan': 'Acarnan', 'Acastus': 'Acastus', 'Acerbas': 'Acerbas', 'Sychaeus': 'Acerbas', 'Acestes': 'Acestes', 'Agestas': 'Acestes', 'Achates': 'Achates', 'Achelous': 'Achelous', 'Achilaos': 'Achelous', 'Acheloüs': 'Achelous', 'Acheloos': 'Achelous', 'Acheron': 'Acheron', 'Achilles': 'Achilles', 'Pelides': 'Achilles', 'Peleiades': 'Achilles', 'Achilleus': 'Achilles', 'Acis': 'Acis', 'Acontius': 'Acontius', 'Acrisius': 'Acrisius', 'Akrisios': 'Acrisius', 'Actaeon': 'Actaeon', 'Aktaion': 'Actaeon', 'Actæon': 'Actaeon', 'Actor': 'Actor', 'Admetus': 'Admetus', 'Adrastus': 'Adrastus', 'Aeacus': 'Aeacus', 'Eacus': 'Aeacus', 'Æacus': 'Aeacus', 'Aegiale': 'Aegiale', 'Euryaleia': 'Aegiale', 'Aigialea': 'Aegiale', 'Aegina'

In [52]:
import numpy as np

# Replace invalid fathers with NaN
unique_characters.loc[
    ~unique_characters["Father"].isin(unique_characters["name"].dropna()), 
    "Father"
] = np.nan

# Replace invalid mothers with NaN
unique_characters.loc[
    ~unique_characters["Mother"].isin(unique_characters["name"].dropna()), 
    "Mother"
] = np.nan

missing_fathers = unique_characters.loc[~unique_characters["Father"].isin(unique_characters["name"].dropna()), "Father"].unique()

# Check mothers
missing_mothers = unique_characters.loc[~unique_characters["Mother"].isin(unique_characters["name"].dropna()), "Mother"].unique()


print("Fathers not in name:", missing_fathers)
print("Mothers not in name:", missing_mothers)

unique_characters['gender'] = unique_characters['gender'].replace('male organism', 'male')
unique_characters['gender'] = unique_characters['gender'].replace('female organism', 'female')

unique_characters.to_csv('datasets/unique_characters.csv', index=False)
unique_characters

Fathers not in name: [nan]
Mothers not in name: [nan]


,name,description,domain,gender,aliases,level1,level2,level3,level4,Title,Father,Mother
0,Abas,"mythical character, son of Lynceus",[],male,"[Abas, son of Lynceus, son of lynceus, son of ...",NaN,NaN,NaN,NaN,King (of Argos),Lynceus,Hypermnestra
1,Abderus,son of Hermes in Greek mythology,[],male,[Abderos],NaN,NaN,NaN,NaN,NaN,Hermes,NaN
2,Absyrtus,mythical brother of Medea,[],male,"[Apsyrtus, Aegialeus]",NaN,NaN,NaN,NaN,NaN,Aeëtes,NaN
3,Acacallis,daughter of Minos in Greek mythology,[],female,[Acacallis],NaN,NaN,NaN,NaN,NaN,Minos,NaN
4,Acamas,son of Theseus,[],male,[],NaN,NaN,NaN,NaN,NaN,Theseus,Phaedra
...,...,...,...,...,...,...,...,...,...,...,...,...
441,Uranus,"primordial Greek deity, god of the Sky",['sky'],male,"[Uranos, Ouranos, Coelus, Caelum, Caelus]",Greek deity,Greek primordial deity,NaN,NaN,NaN,NaN,Gaia
442,Zelus,ancient Greek personification of emulation/envy,"['emulation', 'envy','jealousy']",male,[],Greek deity,allegorical Greek deity,NaN,NaN,NaN,Pallas,Styx
443,Zephyrus,"Greek god of the wind from the west, spring, f...",['wind'],male,[Zephyr],Greek deity,Anemoi,NaN,NaN,NaN,Astraeus,Eos
444,Zethos,king in Greek mythology,[],male,[Zethus],NaN,NaN,NaN,NaN,NaN,Zeus,Hippolyta


In [53]:
# Step 1: Build a set of valid names
valid_names = set(unique_characters["name"])

# Step 2: Keep only rows where both source and target exist in valid_names
edges_filtered = edges[
    edges["source"].isin(valid_names) & edges["target"].isin(valid_names)
].reset_index(drop=True)

edges_filtered.to_csv('datasets/edges_filtered.csv', index=False)
edges_filtered

,source,target,normalized_weight,author,title,period
0,Aphrodite,Cronus,0.005291,Nonnus,Dionysiaca Bks 1 - 14,Late Roman
1,Aphrodite,Nereus,0.005291,Nonnus,Dionysiaca Bks 1 - 14,Late Roman
2,Aphrodite,Thetis,0.010582,Nonnus,Dionysiaca Bks 1 - 14,Late Roman
3,Aphrodite,Triton,0.010582,Nonnus,Dionysiaca Bks 1 - 14,Late Roman
4,Aphrodite,Zeus,0.015873,Nonnus,Dionysiaca Bks 1 - 14,Late Roman
...,...,...,...,...,...,...
28322,Melicertes,Morpheus,0.006061,Virgil,Eclogues,Imperial Roman
28323,Melicertes,Zeus,0.012121,Virgil,Eclogues,Imperial Roman
28324,Orpheus,Pan,0.006061,Virgil,Eclogues,Imperial Roman
28325,Pallas,Paris,0.006061,Virgil,Eclogues,Imperial Roman


In [54]:
from collections import Counter
unique_characters["domain"] = unique_characters["domain"].apply(ensure_list)
# Flatten all lists in the 'domain' column
all_domains = [item for sublist in unique_characters['domain'] for item in sublist]

# Count frequencies
domain_counts = Counter(all_domains)

# Convert to DataFrame (optional, for nice display)
domain_counts_df = pd.DataFrame(domain_counts.items(), columns=['domain', 'count']).sort_values(by='count', ascending=False)

domain_counts_df.head(20)

,domain,count
6,sea,19
0,river,8
36,nature,6
16,fertility,5
47,death,5
7,destiny,4
22,poetry,4
51,seasons,4
3,wind,3
58,spring,3


In [55]:
from collections import Counter
unique_characters["gender"] = unique_characters["gender"].apply(ensure_list)
# Flatten all lists in the 'domain' column
all_genders = [item for sublist in unique_characters['gender'] for item in sublist]

# Count frequencies
gender_counts = Counter(all_genders)

# Convert to DataFrame (optional, for nice display)
gender_counts_df = pd.DataFrame(gender_counts.items(), columns=['gender', 'count']).sort_values(by='count', ascending=False)
gender_counts_df

,gender,count
0,male,274
1,female,171
